# 06 · Supervised Modeling

Train 5 classifiers + 2 regressors with StratifiedGroupKFold (groups = `disaster_number`) and SMOTE in an imblearn pipeline. Tune RF with GridSearchCV, XGB with Optuna. Evaluate on the VAL split.

In [1]:
import sys, os
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 60)
from config import (
    DATA_PATHS, HURRICANE_META, STATES_IN_SCOPE,
    TARGET_COL, TARGET_CLASS_COL, FEATURE_GROUPS,
    RANDOM_STATE, SEVERITY_BINS, SEVERITY_LABELS,
)
RAW = DATA_PATHS['raw']; INTERIM = DATA_PATHS['interim']
PROC = DATA_PATHS['processed']; MODELS = DATA_PATHS['models']
OUT = DATA_PATHS['outputs']


In [2]:
import joblib
from sklearn.preprocessing import LabelEncoder
from src.modeling import (
    build_preprocessor, build_pipeline, get_classifiers, get_regressors,
    cv_score, tune_rf_grid, tune_xgb_optuna, regression_eval, classification_eval,
)
from config import CONTINUOUS_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES
df = pd.read_csv(PROC / 'abt_with_clusters.csv', dtype={'zip_code': str}).dropna(subset=[TARGET_CLASS_COL])
features = (FEATURE_GROUPS['demographics']+FEATURE_GROUPS['svi']
    +FEATURE_GROUPS['food_access']+FEATURE_GROUPS['flood']
    +FEATURE_GROUPS['storm']+['cluster_label','state'])
features = [f for f in features if f in df.columns]
X, y, g = df[features], df[TARGET_CLASS_COL], df['disaster_number']
le = LabelEncoder().fit(SEVERITY_LABELS); y_enc = le.transform(y)
tr = df['train_test_split']=='TRAIN'; va = df['train_test_split']=='VAL'

# Filter preprocessor feature lists to columns actually present in X
avail = set(X.columns)
cont = [c for c in CONTINUOUS_FEATURES if c in avail]
if 'cluster_label' in avail and 'cluster_label' not in cont:
    cont.append('cluster_label')
binf = [c for c in BINARY_FEATURES if c in avail]
catf = [c for c in CATEGORICAL_FEATURES if c in avail]
pre = build_preprocessor(continuous=cont, categorical=catf, binary=binf)
print('using', len(cont), 'cont +', len(catf), 'cat +', len(binf), 'bin features')

using 21 cont + 2 cat + 1 bin features


## Cross-val on TRAIN split

In [ ]:
results = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre)
    mean, arr = cv_score(pipe, X[tr], pd.Series(y_enc[tr]), g[tr])
    results[name] = mean
    print(f'{name:6s}  CV F1-weighted = {mean:.3f}  (folds: {arr.round(3)})')

C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\chait\AppData\Roaming\Python\Python314\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


rf      CV F1-weighted = 0.339  (folds: [0.479 0.267 0.271])


## Tune RF + XGBoost

In [ ]:
rf_gs = tune_rf_grid(X[tr], pd.Series(y_enc[tr]), g[tr], preprocessor=pre)
print('RF best:', rf_gs.best_params_, rf_gs.best_score_)

In [ ]:
try:
    study = tune_xgb_optuna(X[tr], pd.Series(y_enc[tr]), g[tr], n_trials=50, preprocessor=pre)
    print('XGB best:', study.best_params, study.best_value)
except Exception as e:
    print('XGB tuning skipped:', e); study = None

## Fit on TRAIN, evaluate on VAL — WITH cluster_label

In [ ]:
val_scores = {}
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre).fit(X[tr], y_enc[tr])
    pred = pipe.predict(X[va])
    val_scores[name+'_withcluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
    print(f'{name} WITH cluster  F1={val_scores[name+"_withcluster"]:.3f}')

## Same evaluation WITHOUT cluster_label

In [ ]:
features_noc = [f for f in features if f != 'cluster_label']
cont_noc = [c for c in cont if c != 'cluster_label' and c in features_noc]
pre_noc = build_preprocessor(continuous=cont_noc, categorical=catf, binary=binf)
for name, clf in get_classifiers().items():
    pipe = build_pipeline(clf, preprocessor=pre_noc).fit(X[tr][features_noc], y_enc[tr])
    pred = pipe.predict(X[va][features_noc])
    val_scores[name+'_nocluster'] = classification_eval(y_enc[va], pred)['f1_weighted']
cmp = pd.DataFrame({
  'with_cluster': {k.replace('_withcluster',''): v for k,v in val_scores.items() if k.endswith('_withcluster')},
  'without_cluster': {k.replace('_nocluster',''): v for k,v in val_scores.items() if k.endswith('_nocluster')},
}); cmp['delta'] = cmp['with_cluster'] - cmp['without_cluster']; cmp

## Regression variants

In [ ]:
from sklearn.pipeline import Pipeline as SkPipeline
regs_scores = {}
for name, reg in get_regressors().items():
    pipe_r = SkPipeline([('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)), ('reg', reg)])
    pipe_r.fit(X[tr], df.loc[tr, TARGET_COL])
    pred = pipe_r.predict(X[va])
    regs_scores[name] = regression_eval(df.loc[va, TARGET_COL], pred)
regs_scores

## Save best classifier + regressor

In [ ]:
best_name = max({k:v for k,v in val_scores.items() if k.endswith('_withcluster')}.items(), key=lambda kv: kv[1])[0].replace('_withcluster','')
print('best classifier:', best_name)
best_clf = build_pipeline(get_classifiers()[best_name], preprocessor=pre).fit(X[tr], y_enc[tr])
joblib.dump({'pipe': best_clf, 'label_encoder': le, 'features': features},
            MODELS / 'best_classifier.pkl')
best_reg_name = min(regs_scores.items(), key=lambda kv: kv[1]['rmse'])[0]
best_reg = SkPipeline(
    [('pre', build_preprocessor(continuous=cont, categorical=catf, binary=binf)),
     ('reg', get_regressors()[best_reg_name])])
best_reg.fit(X[tr], df.loc[tr, TARGET_COL])
joblib.dump({'pipe': best_reg, 'features': features}, MODELS / 'best_regressor.pkl')
print('saved', MODELS / 'best_classifier.pkl', '/', MODELS / 'best_regressor.pkl')